# Desenvolvimento dos merges para predicao de falha

Este notebook prepara apenas os datasets derivados para a etapa futura de modelagem. Ele nao treina modelos e nao altera nenhum CSV original. Todas as saidas sao gravadas em `output/merged_datasets`.


## Estrategia adotada

A melhor configuracao para esta etapa e separar as fontes por confianca:

- **EDP ouro**: base principal supervisionada, pois possui SCADA, logs e arquivo explicito de falhas.
- **Penmanshiel fraco**: base auxiliar, usando status operacionais para criar eventos candidatos a falha.
- **Schema comum EDP + Penmanshiel**: base reduzida para comparar dominios e preparar experimentos futuros.

Essa separacao evita tratar status fraco do Penmanshiel como se fosse o mesmo ground truth do EDP.


## Artefatos gerados

- `edp_gold_scada_failures_logs_metmast_empirical.csv`
- `penmanshiel_scada_status_weak_labels.csv`
- `penmanshiel_status_failure_candidates.csv`
- `edp_penmanshiel_common_schema.csv`
- `merge_artifacts_summary.csv`


In [10]:
from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd

from wind_dashboard_core.loaders import load_dataset

ROOT = Path.cwd().resolve()
OUT_DIR = ROOT / "output" / "merged_datasets"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EDP_SCADA_PATH = ROOT / "Wind_Turbine_SCADA_tratado_consolidado.csv"
EDP_FAILURES_PATH = ROOT / "opendata-wind-failures-2017 (3).csv"
EDP_LOGS_PATH = ROOT / "Wind%20Turbines%20Logs%202017 (3).csv"
METMAST_PATH = ROOT / "Onsite_MetMast_SCADA_tratado.csv"

PREDICTION_WINDOWS_H = [1, 6, 12, 24, 48, 72, 168]
LOG_WINDOWS_H = [1, 6, 24, 72]

OUT_DIR


WindowsPath('C:/Users/enzob/OneDrive/Documents/IC_ebr/output/merged_datasets')

## Funcoes auxiliares

As funcoes abaixo padronizam timestamps, criam rotulos `falha nos proximos Xh` e agregam logs anteriores ao timestamp. A regra principal e nao usar informacao futura como feature.


In [11]:
def parse_timestamp(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(None)


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(col).strip() for col in out.columns]
    return out


def add_next_event_labels(base, events, entity_col, timestamp_col, event_time_col, prefix, windows_h, event_cols=None):
    out = base.copy()
    event_cols = event_cols or []
    out[f"{prefix}_next_timestamp"] = pd.NaT
    out[f"{prefix}_time_to_next_h"] = np.nan
    for col in event_cols:
        out[f"{prefix}_next_{col}"] = pd.NA
    for hours in windows_h:
        out[f"{prefix}_next_{hours}h"] = 0

    out["_orig_order"] = np.arange(len(out))
    for entity, idx in out.groupby(entity_col, observed=True).groups.items():
        base_idx = np.asarray(list(idx))
        base_times = pd.to_datetime(out.loc[base_idx, timestamp_col], errors="coerce").to_numpy(dtype="datetime64[ns]")
        ev = events[events[entity_col] == entity].dropna(subset=[event_time_col]).sort_values(event_time_col)
        if ev.empty:
            continue

        ev_times = pd.to_datetime(ev[event_time_col], errors="coerce").to_numpy(dtype="datetime64[ns]")
        pos = np.searchsorted(ev_times, base_times, side="left")
        valid = pos < len(ev_times)
        if not valid.any():
            continue

        selected_idx = base_idx[valid]
        selected_pos = pos[valid]
        next_times = ev_times[selected_pos]
        delta_h = (next_times - base_times[valid]) / np.timedelta64(1, "h")

        out.loc[selected_idx, f"{prefix}_next_timestamp"] = pd.to_datetime(next_times)
        out.loc[selected_idx, f"{prefix}_time_to_next_h"] = delta_h.astype(float)
        for hours in windows_h:
            out.loc[selected_idx, f"{prefix}_next_{hours}h"] = ((delta_h >= 0) & (delta_h <= hours)).astype(int)
        for col in event_cols:
            values = ev[col].reset_index(drop=True).iloc[selected_pos].to_numpy()
            out.loc[selected_idx, f"{prefix}_next_{col}"] = values

    return out.sort_values("_orig_order").drop(columns=["_orig_order"])


def rolling_log_features(base: pd.DataFrame, logs: pd.DataFrame) -> pd.DataFrame:
    out = base.copy()
    for hours in LOG_WINDOWS_H:
        out[f"logs_count_{hours}h"] = 0
    out["time_since_last_log_h"] = np.nan
    out["last_log_timestamp"] = pd.NaT
    out["last_log_remark"] = pd.NA

    category_patterns = {
        "external_power_ref": r"external power ref",
        "temperature_or_cooling": r"temp|vent|cool|fan|gear.*oil|hot hv",
        "yaw": r"yaw",
        "generator": r"gen\.|generator",
        "pitch": r"pitch|blade",
        "grid_or_power": r"grid|power|trafo|voltage|current",
    }
    for category in category_patterns:
        out[f"logs_{category}_24h"] = 0
        out[f"logs_{category}_72h"] = 0

    logs = logs.dropna(subset=["entity", "log_timestamp"]).sort_values(["entity", "log_timestamp"]).copy()
    logs["remark_text"] = logs["Remark"].fillna("").astype(str)

    for entity, idx in out.groupby("entity", observed=True).groups.items():
        base_idx = np.asarray(list(idx))
        base_times = pd.to_datetime(out.loc[base_idx, "timestamp"], errors="coerce").to_numpy(dtype="datetime64[ns]")
        entity_logs = logs[logs["entity"] == entity].sort_values("log_timestamp")
        if entity_logs.empty:
            continue

        log_times = pd.to_datetime(entity_logs["log_timestamp"], errors="coerce").to_numpy(dtype="datetime64[ns]")
        right = np.searchsorted(log_times, base_times, side="right")
        for hours in LOG_WINDOWS_H:
            left = np.searchsorted(log_times, base_times - np.timedelta64(hours, "h"), side="left")
            out.loc[base_idx, f"logs_count_{hours}h"] = (right - left).astype(int)

        previous = right - 1
        has_previous = previous >= 0
        if has_previous.any():
            selected_idx = base_idx[has_previous]
            selected_previous = previous[has_previous]
            last_times = log_times[selected_previous]
            delta_h = (base_times[has_previous] - last_times) / np.timedelta64(1, "h")
            out.loc[selected_idx, "time_since_last_log_h"] = delta_h.astype(float)
            out.loc[selected_idx, "last_log_timestamp"] = pd.to_datetime(last_times)
            out.loc[selected_idx, "last_log_remark"] = entity_logs["remark_text"].reset_index(drop=True).iloc[selected_previous].to_numpy()

        for category, pattern in category_patterns.items():
            cat_logs = entity_logs[entity_logs["remark_text"].str.contains(pattern, case=False, regex=True, na=False)]
            if cat_logs.empty:
                continue
            cat_times = pd.to_datetime(cat_logs["log_timestamp"], errors="coerce").to_numpy(dtype="datetime64[ns]")
            cat_right = np.searchsorted(cat_times, base_times, side="right")
            for hours in [24, 72]:
                cat_left = np.searchsorted(cat_times, base_times - np.timedelta64(hours, "h"), side="left")
                out.loc[base_idx, f"logs_{category}_{hours}h"] = (cat_right - cat_left).astype(int)

    return out


## Merge 1: EDP ouro

Combina SCADA EDP, falhas reais, logs em janelas moveis, MetMast e residual de curva empirica EDP (~2 MW). A referencia agora e uma curva empirica construida com dados saudaveis do proprio EDP, que representa melhor turbinas de aproximadamente 2 MW. Este e o dataset mais confiavel para uma futura modelagem supervisionada.


In [12]:
def _first_valid(series: pd.Series) -> object:
    valid = series.dropna()
    if valid.empty:
        return np.nan
    return valid.iloc[0]


def _series_signature(series: pd.Series) -> tuple[int, str]:
    values = series.astype("string").fillna("<NA>")
    hashes = pd.util.hash_pandas_object(values, index=False).to_numpy(dtype="uint64")
    digest = hashlib.blake2b(hashes.tobytes(), digest_size=16).hexdigest()
    return len(series), digest


def drop_exact_duplicate_columns(df: pd.DataFrame, protected_cols: set[str] | None = None) -> pd.DataFrame:
    """Remove columns with exactly the same values, keeping the first occurrence."""
    protected_cols = protected_cols or set()
    signatures: dict[tuple[int, str], str] = {}
    drop_cols: list[str] = []
    comparable = df.astype("string").fillna("<NA>")

    for col in df.columns:
        if col in protected_cols:
            continue
        signature = _series_signature(df[col])
        previous = signatures.get(signature)
        if previous is not None and comparable[previous].equals(comparable[col]):
            drop_cols.append(col)
        else:
            signatures[signature] = col

    return df.drop(columns=drop_cols)


def aggregate_edp_entity_timestamp(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse duplicated EDP rows at the intended entity+timestamp grain.

    The duplicated source rows are pairs, so numeric median is equal to mean for
    these conflicts while being safer if a future group has more than two rows.
    """
    key_cols = ["dataset", "label_source", "entity", "timestamp"]
    if not set(key_cols).issubset(df.columns):
        return df

    label_pattern = re.compile(r"failure_next_\d+h$")
    agg_map = {}
    for col in df.columns:
        if col in key_cols:
            continue
        if label_pattern.match(col):
            agg_map[col] = "max"
        elif pd.api.types.is_bool_dtype(df[col]):
            agg_map[col] = "max"
        elif pd.api.types.is_numeric_dtype(df[col]):
            agg_map[col] = "median"
        else:
            agg_map[col] = _first_valid

    out = (
        df.groupby(key_cols, observed=True, dropna=False, sort=False)
        .agg(agg_map)
        .reset_index()
    )
    return out.loc[:, df.columns]


def clean_edp_gold_output(df: pd.DataFrame) -> pd.DataFrame:
    protected = {"dataset", "label_source", "entity", "timestamp"}
    return drop_exact_duplicate_columns(df, protected_cols=protected)


def add_edp_empirical_reference_features(df: pd.DataFrame, bin_width: float = 0.5, min_bin_count: int = 30) -> pd.DataFrame:
    """Add a 2 MW-compatible empirical power reference from healthy EDP rows.

    This replaces the old NREL 5 MW reference. For final model validation, rebuild
    this curve inside each training split to avoid temporal leakage.
    """
    out = df.copy()
    wind = pd.to_numeric(out["Amb_WindSpeed_Avg"], errors="coerce")
    power_norm = pd.to_numeric(out["power_norm"], errors="coerce")
    produced_power = pd.to_numeric(out["Grd_Prod_Pwr_Avg"], errors="coerce")

    healthy_mask = wind.notna() & power_norm.notna() & produced_power.ge(0) & power_norm.between(0, 1.25)
    if "outlier_flag" in out.columns:
        healthy_mask &= ~out["outlier_flag"].astype(str).str.lower().isin(["true", "1", "yes"])
    if "failure_next_168h" in out.columns:
        healthy_mask &= pd.to_numeric(out["failure_next_168h"], errors="coerce").fillna(0).eq(0)

    healthy = out.loc[healthy_mask, ["entity", "Amb_WindSpeed_Avg", "power_norm"]].copy()
    healthy["wind_bin"] = (np.floor(pd.to_numeric(healthy["Amb_WindSpeed_Avg"], errors="coerce") / bin_width) * bin_width).round(3)

    global_curve = (
        healthy.groupby("wind_bin", observed=True)["power_norm"]
        .agg(reference_power_norm="median", count="count")
        .reset_index()
        .query("count >= @min_bin_count")
        .sort_values("wind_bin")
    )

    out["edp_empirical_reference_power_norm"] = np.nan
    for entity, idx in out.groupby("entity", observed=True).groups.items():
        entity_curve = (
            healthy[healthy["entity"] == entity]
            .groupby("wind_bin", observed=True)["power_norm"]
            .agg(reference_power_norm="median", count="count")
            .reset_index()
            .query("count >= @min_bin_count")
            .sort_values("wind_bin")
        )
        curve = entity_curve if len(entity_curve) >= 4 else global_curve
        if curve.empty:
            continue
        x = curve["wind_bin"].to_numpy(dtype=float)
        y = curve["reference_power_norm"].to_numpy(dtype=float)
        ws = pd.to_numeric(out.loc[idx, "Amb_WindSpeed_Avg"], errors="coerce").to_numpy(dtype=float)
        out.loc[idx, "edp_empirical_reference_power_norm"] = np.interp(ws, x, y, left=0.0, right=float(y[-1]))

    out["edp_empirical_power_residual_norm"] = out["power_norm"] - out["edp_empirical_reference_power_norm"]
    return out


def build_edp_gold() -> pd.DataFrame:
    edp = clean_columns(pd.read_csv(EDP_SCADA_PATH, low_memory=False))
    edp["entity"] = edp["Turbine_ID"].astype(str).str.strip()
    edp["timestamp"] = parse_timestamp(edp["Timestamp"])
    edp["dataset"] = "edp"
    edp["label_source"] = "edp_failure_file"

    rated = edp.groupby("entity")["Grd_Prod_Pwr_Avg"].transform(lambda s: pd.to_numeric(s, errors="coerce").clip(lower=0).quantile(0.995))
    edp["rated_power_est_kw"] = rated.replace(0, np.nan)
    edp["power_norm"] = pd.to_numeric(edp["Grd_Prod_Pwr_Avg"], errors="coerce") / edp["rated_power_est_kw"]
    edp["yaw_error_deg"] = ((pd.to_numeric(edp["Nac_Direction_Avg"], errors="coerce") - pd.to_numeric(edp["Amb_WindDir_Abs_Avg"], errors="coerce") + 180) % 360) - 180

    failures = pd.read_csv(EDP_FAILURES_PATH, sep=";", low_memory=False)
    failures = clean_columns(failures).rename(columns={"Turbine_ID": "entity", "Timestamp": "failure_timestamp"})
    failures["entity"] = failures["entity"].astype(str).str.strip()
    failures["failure_timestamp"] = parse_timestamp(failures["failure_timestamp"])
    failures = failures[failures["entity"].isin(edp["entity"].unique())].copy()

    edp = add_next_event_labels(
        edp, failures, "entity", "timestamp", "failure_timestamp", "failure",
        PREDICTION_WINDOWS_H, event_cols=["Component", "Remarks"]
    )

    logs = pd.read_csv(EDP_LOGS_PATH, sep=";", low_memory=False)
    logs = clean_columns(logs).rename(columns={"Turbine_Identifier": "entity", "Time_Detected": "log_timestamp"})
    logs["entity"] = logs["entity"].astype(str).str.strip()
    logs["log_timestamp"] = parse_timestamp(logs["log_timestamp"])
    edp = rolling_log_features(edp, logs)

    met = clean_columns(pd.read_csv(METMAST_PATH, low_memory=False))
    met["timestamp"] = parse_timestamp(met["Timestamp"])
    met = met.drop(columns=["Timestamp"], errors="ignore")
    met = met.rename(columns={col: f"met_{col}" for col in met.columns if col != "timestamp"})
    edp = edp.merge(met, on="timestamp", how="left")

    edp = add_edp_empirical_reference_features(edp)
    edp = edp.drop(columns=["Turbine_ID", "Timestamp"], errors="ignore")
    edp = aggregate_edp_entity_timestamp(edp)
    edp = clean_edp_gold_output(edp)
    front_cols = [
        "dataset", "label_source", "entity", "timestamp",
        "failure_next_timestamp", "failure_time_to_next_h", "failure_next_Component", "failure_next_Remarks",
        "failure_next_24h", "failure_next_72h", "failure_next_168h",
        "power_norm", "edp_empirical_reference_power_norm", "edp_empirical_power_residual_norm", "rated_power_est_kw", "yaw_error_deg",
    ]
    front_cols = [col for col in front_cols if col in edp.columns]
    return edp.loc[:, front_cols + [col for col in edp.columns if col not in front_cols]]


## Merge 2: Penmanshiel com rotulos fracos

Os arquivos `Status_Penmanshiel_*` sao filtrados para eventos candidatos a falha. Eventos de vento, manutencao programada, comunicacao e paradas manuais sao excluidos da definicao de falha interna da turbina.


In [13]:
def classify_penmanshiel_status_events(status: pd.DataFrame) -> pd.DataFrame:
    out = status.copy()
    service = out["Service contract category"].astype("string")
    iec = out["IEC category"].astype("string")
    status_label = out["Status"].astype("string")
    message = out["Message"].astype("string")

    include_service = service.str.contains(
        "Generator and Converter errors|Mechanical error|Electrical error|Pitch errors|Safety chain|Safety stop|Sensor error|Repeated error|Warnings",
        case=False, na=False,
    )
    include_iec = iec.str.contains("Forced outage|Out of Electrical Specification|Partial Performance", case=False, na=False)
    exclude = (
        service.str.contains("External stop|Manual stop|Remote stop|Operating states|System OK|WEC Shutdown", case=False, na=False)
        | iec.str.contains("Out of Environmental Specification|Scheduled Maintenance|Requested Shutdown|Technical Standby|Full Performance", case=False, na=False)
        | status_label.str.contains("Communication", case=False, na=False)
        | message.str.contains("Wind < start|Max[.] wind speed|Manual|Externally stopped|Park master|Battery test|Cable autounwind|Hydraulic oil flushing", case=False, na=False)
    )
    out["weak_failure_event"] = ((include_service | include_iec) & ~exclude).astype(int)

    conditions = [
        service.str.contains("Generator and Converter errors", case=False, na=False) | message.str.contains("generator|converter|frequency converter", case=False, na=False),
        service.str.contains("Mechanical error", case=False, na=False) | message.str.contains("gear|brake|oil|mechanical", case=False, na=False),
        service.str.contains("Electrical error", case=False, na=False) | iec.str.contains("Electrical", case=False, na=False) | message.str.contains("grid|ups|electrical|voltage", case=False, na=False),
        service.str.contains("Pitch errors", case=False, na=False) | message.str.contains("pitch|axis|blade", case=False, na=False),
        service.str.contains("Safety chain|Safety stop", case=False, na=False) | message.str.contains("safety|tower oscillation", case=False, na=False),
        service.str.contains("Sensor error", case=False, na=False) | message.str.contains("sensor|implausible|anemometer|vane", case=False, na=False),
        service.str.contains("Warnings", case=False, na=False),
    ]
    labels = ["generator_converter", "mechanical", "electrical", "pitch", "safety", "sensor", "warning"]
    out["weak_failure_type"] = np.select(conditions, labels, default="other")
    out.loc[out["weak_failure_event"] == 0, "weak_failure_type"] = "not_failure_candidate"
    return out


def read_penmanshiel_status_events() -> pd.DataFrame:
    frames = []
    for path in sorted(ROOT.glob("datasets*-penmanshiel/Status_Penmanshiel_*.csv")):
        df = pd.read_csv(path, comment="#", na_values=["-", "NaN", ""], low_memory=False)
        df = clean_columns(df)
        match = re.search(r"Penmanshiel_(\d{2})_", path.name)
        turbine_id = match.group(1) if match else path.stem
        df["entity"] = f"Penmanshiel {turbine_id}"
        df["status_file"] = path.name
        frames.append(df)
    status = pd.concat(frames, ignore_index=True, sort=False)
    status = status.rename(columns={"Timestamp start": "status_timestamp", "Timestamp end": "status_end"})
    status["status_timestamp"] = pd.to_datetime(status["status_timestamp"], errors="coerce")
    status["status_end"] = pd.to_datetime(status["status_end"], errors="coerce")
    return classify_penmanshiel_status_events(status)


def build_penmanshiel_weak() -> tuple[pd.DataFrame, pd.DataFrame]:
    pen = load_dataset("penmanshiel", ROOT).copy()
    pen["timestamp"] = pd.to_datetime(pen["timestamp"], errors="coerce")
    pen["dataset"] = "penmanshiel"
    pen["label_source"] = "penmanshiel_status_weak"

    status = read_penmanshiel_status_events()
    candidates = status[status["weak_failure_event"] == 1].dropna(subset=["status_timestamp"]).copy()
    candidates = candidates.drop_duplicates().drop(columns=["Comment", "weak_failure_event"], errors="ignore")
    event_cols = ["Status", "Code", "Message", "Service contract category", "IEC category", "weak_failure_type"]
    pen = add_next_event_labels(
        pen, candidates, "entity", "timestamp", "status_timestamp", "weak_failure",
        [24, 72, 168], event_cols=event_cols
    )

    service = pen.get("service_contract_category", pd.Series(pd.NA, index=pen.index)).astype("string")
    iec = pen.get("iec_category", pd.Series(pd.NA, index=pen.index)).astype("string")
    status_label = pen.get("status", pd.Series(pd.NA, index=pen.index)).astype("string")
    message = pen.get("status_message", pd.Series(pd.NA, index=pen.index)).astype("string")
    current_include = service.str.contains(
        "Generator and Converter errors|Mechanical error|Electrical error|Pitch errors|Safety chain|Safety stop|Sensor error|Repeated error|Warnings",
        case=False, na=False,
    ) | iec.str.contains("Forced outage|Out of Electrical Specification|Partial Performance", case=False, na=False)
    current_exclude = (
        service.str.contains("External stop|Manual stop|Remote stop|Operating states|System OK", case=False, na=False)
        | iec.str.contains("Out of Environmental Specification|Scheduled Maintenance|Requested Shutdown|Technical Standby|Full Performance", case=False, na=False)
        | status_label.str.contains("Communication", case=False, na=False)
        | message.str.contains("Wind < start|Max[.] wind speed|Manual|Externally stopped|Park master|Battery test|Cable autounwind|Hydraulic oil flushing", case=False, na=False)
    )
    pen["current_status_failure_like"] = (current_include & ~current_exclude).astype(int)
    return pen, candidates


## Merge 3: schema comum EDP + Penmanshiel

Este arquivo reduz EDP e Penmanshiel para colunas compat?veis. Ele nao deve substituir o EDP ouro, mas ajuda em comparacoes e experimentos de transferencia.


In [14]:
def build_common_dataset(edp_gold: pd.DataFrame, pen_weak: pd.DataFrame) -> pd.DataFrame:
    edp_common = pd.DataFrame({
        "dataset": "edp",
        "label_source": edp_gold["label_source"],
        "entity": edp_gold["entity"],
        "timestamp": edp_gold["timestamp"],
        "wind_speed_ms": edp_gold["Amb_WindSpeed_Avg"],
        "power_kw": edp_gold["Grd_Prod_Pwr_Avg"],
        "power_norm": edp_gold["power_norm"],
        "wind_direction_deg": edp_gold["Amb_WindDir_Abs_Avg"],
        "nac_direction_deg": edp_gold["Nac_Direction_Avg"],
        "yaw_error_deg": edp_gold["yaw_error_deg"],
        "wind_speed_max_ms": edp_gold["Amb_WindSpeed_Max"],
        "temperature_c": edp_gold["Amb_Temp_Avg"],
        "rotor_speed_rpm": edp_gold["Rtr_RPM_Avg"],
        "generator_rpm": edp_gold["Gen_RPM_Avg"],
        "pitch_angle_deg": edp_gold["Blds_PitchAngle_Avg"],
        "power_residual_norm": edp_gold["edp_empirical_power_residual_norm"],
        "logs_count_24h": edp_gold["logs_count_24h"],
        "failure_next_24h": edp_gold["failure_next_24h"],
        "failure_next_72h": edp_gold["failure_next_72h"],
        "failure_next_168h": edp_gold["failure_next_168h"],
        "next_failure_timestamp": edp_gold["failure_next_timestamp"],
        "next_failure_type": edp_gold["failure_next_Component"],
        "current_status_failure_like": pd.NA,
    })
    pen_common = pd.DataFrame({
        "dataset": "penmanshiel",
        "label_source": pen_weak["label_source"],
        "entity": pen_weak["entity"],
        "timestamp": pen_weak["timestamp"],
        "wind_speed_ms": pen_weak["wind_speed_ms"],
        "power_kw": pen_weak["power_kw"],
        "power_norm": pen_weak["power_norm"],
        "wind_direction_deg": pen_weak["wind_direction_deg"],
        "nac_direction_deg": pen_weak["nac_direction_deg"],
        "yaw_error_deg": pen_weak["yaw_error"],
        "wind_speed_max_ms": pen_weak["wind_speed_max_ms"],
        "temperature_c": pen_weak["temperature_c"],
        "rotor_speed_rpm": pen_weak["rotor_speed_rpm"],
        "generator_rpm": pen_weak["generator_rpm"],
        "pitch_angle_deg": pen_weak["pitch_angle_deg"],
        "power_residual_norm": pd.NA,
        "logs_count_24h": pd.NA,
        "failure_next_24h": pen_weak["weak_failure_next_24h"],
        "failure_next_72h": pen_weak["weak_failure_next_72h"],
        "failure_next_168h": pen_weak["weak_failure_next_168h"],
        "next_failure_timestamp": pen_weak["weak_failure_next_timestamp"],
        "next_failure_type": pen_weak["weak_failure_next_weak_failure_type"],
        "current_status_failure_like": pen_weak["current_status_failure_like"],
    })
    return pd.concat([edp_common, pen_common], ignore_index=True, sort=False)


def build_all_exports() -> pd.DataFrame:
    edp_gold = build_edp_gold()
    edp_gold.to_csv(OUT_DIR / "edp_gold_scada_failures_logs_metmast_empirical.csv", index=False)

    pen_weak, pen_candidates = build_penmanshiel_weak()
    pen_weak.to_csv(OUT_DIR / "penmanshiel_scada_status_weak_labels.csv", index=False)
    pen_candidates.to_csv(OUT_DIR / "penmanshiel_status_failure_candidates.csv", index=False)

    common = build_common_dataset(edp_gold, pen_weak)
    common.to_csv(OUT_DIR / "edp_penmanshiel_common_schema.csv", index=False)

    summary_rows = []
    for name, df in [
        ("edp_gold_scada_failures_logs_metmast_empirical", edp_gold),
        ("penmanshiel_scada_status_weak_labels", pen_weak),
        ("penmanshiel_status_failure_candidates", pen_candidates),
        ("edp_penmanshiel_common_schema", common),
    ]:
        row = {"artifact": name, "rows": len(df), "columns": len(df.columns)}
        for col in [
            "failure_next_24h", "failure_next_72h", "failure_next_168h",
            "weak_failure_next_24h", "weak_failure_next_72h", "weak_failure_next_168h",
            "current_status_failure_like",
        ]:
            if col in df.columns:
                row[f"{col}_positive"] = int(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())
        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(OUT_DIR / "merge_artifacts_summary.csv", index=False)
    return summary


## Executar os merges

A celula abaixo gera todos os CSVs derivados. Pode demorar alguns minutos por causa do tamanho dos arquivos SCADA.


In [15]:
summary = build_all_exports()
summary


C:\Users\enzob\AppData\Local\Temp\ipykernel_13464\2190631802.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([edp_common, pen_common], ignore_index=True, sort=False)


,artifact,rows,columns,failure_next_24h_positive,failure_next_72h_positive,failure_next_168h_positive,weak_failure_next_24h_positive,weak_failure_next_72h_positive,weak_failure_next_168h_positive,current_status_failure_like_positive
0,edp_gold_scada_failures_logs_metmast_empirical,209236,162,1295.0,3641.0,8248.0,NaN,NaN,NaN,NaN
1,penmanshiel_scada_status_weak_labels,737856,50,NaN,NaN,NaN,55156.0,135993.0,262622.0,5102.0
2,penmanshiel_status_failure_candidates,1267,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,edp_penmanshiel_common_schema,947092,23,56451.0,139634.0,270870.0,NaN,NaN,NaN,5102.0


## Auditoria rapida

Depois de executar os merges, confira tamanhos dos arquivos e contagens de positivos.


In [16]:
artifacts = sorted(OUT_DIR.glob("*.csv"))
for path in artifacts:
    print(f"{path.name}: {path.stat().st_size / 1_000_000:.1f} MB")

summary_check = pd.read_csv(OUT_DIR / "merge_artifacts_summary.csv")
summary_check


edp_gold_scada_failures_logs_metmast_empirical.csv: 191.5 MB
edp_penmanshiel_common_schema.csv: 259.8 MB
merge_artifacts_summary.csv: 0.0 MB
penmanshiel_scada_status_weak_labels.csv: 481.3 MB
penmanshiel_status_failure_candidates.csv: 0.3 MB


,artifact,rows,columns,failure_next_24h_positive,failure_next_72h_positive,failure_next_168h_positive,weak_failure_next_24h_positive,weak_failure_next_72h_positive,weak_failure_next_168h_positive,current_status_failure_like_positive
0,edp_gold_scada_failures_logs_metmast_empirical,209236,162,1295.0,3641.0,8248.0,NaN,NaN,NaN,NaN
1,penmanshiel_scada_status_weak_labels,737856,50,NaN,NaN,NaN,55156.0,135993.0,262622.0,5102.0
2,penmanshiel_status_failure_candidates,1267,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,edp_penmanshiel_common_schema,947092,23,56451.0,139634.0,270870.0,NaN,NaN,NaN,5102.0


## Amostras dos artefatos

Esta celula le apenas as primeiras linhas de cada CSV, evitando carregar tudo em memoria.


In [17]:
for path in sorted(OUT_DIR.glob("*.csv")):
    sample = pd.read_csv(path, nrows=5, low_memory=False)
    print(f"\n{path.name} | colunas: {len(sample.columns)}")
    display(sample)



edp_gold_scada_failures_logs_metmast_empirical.csv | colunas: 162


,dataset,label_source,entity,timestamp,failure_next_timestamp,failure_time_to_next_h,failure_next_Component,failure_next_Remarks,failure_next_24h,failure_next_72h,...,met_Anemometer2_Freq,met_Anemometer2_Offset,met_Anemometer2_CorrGain,met_Anemometer2_CorrOffset,met_DistanceAirPress,met_AirRessureSensorZeroOffset,met_Anemometer1_Avg_Freq,met_Anemometer2_Avg_Freq,met_Pressure_Avg_Freq,met_outlier_flag
0,edp,edp_failure_file,T06,2017-12-29 20:30:00,NaN,NaN,NaN,NaN,0,0,...,0.0499,0.24,1.0,0.0,0.0,600.0,66.0,65.0,420.0,False
1,edp,edp_failure_file,T06,2017-12-29 20:40:00,NaN,NaN,NaN,NaN,0,0,...,0.0499,0.24,1.0,0.0,0.0,600.0,78.0,77.0,420.0,False
2,edp,edp_failure_file,T06,2017-12-29 21:10:00,NaN,NaN,NaN,NaN,0,0,...,0.0499,0.24,1.0,0.0,0.0,600.0,78.0,77.0,420.0,False
3,edp,edp_failure_file,T06,2017-12-29 21:20:00,NaN,NaN,NaN,NaN,0,0,...,0.0499,0.24,1.0,0.0,0.0,600.0,64.0,63.0,420.0,False
4,edp,edp_failure_file,T06,2017-12-29 21:50:00,NaN,NaN,NaN,NaN,0,0,...,0.0499,0.24,1.0,0.0,0.0,600.0,75.0,74.0,420.0,False



edp_penmanshiel_common_schema.csv | colunas: 23


,dataset,label_source,entity,timestamp,wind_speed_ms,power_kw,power_norm,wind_direction_deg,nac_direction_deg,yaw_error_deg,...,generator_rpm,pitch_angle_deg,power_residual_norm,logs_count_24h,failure_next_24h,failure_next_72h,failure_next_168h,next_failure_timestamp,next_failure_type,current_status_failure_like
0,edp,edp_failure_file,T06,2017-12-29 20:30:00,3.2,-10.4,-0.005200,277.8,277.4,-0.4,...,647.1,16.6,-0.017711,324,0,0,0,NaN,NaN,NaN
1,edp,edp_failure_file,T06,2017-12-29 20:40:00,3.4,-9.7,-0.004850,286.5,281.2,-5.3,...,293.1,24.0,-0.023921,323,0,0,0,NaN,NaN,NaN
2,edp,edp_failure_file,T06,2017-12-29 21:10:00,3.5,2.9,0.001450,314.9,314.6,-0.3,...,475.6,19.0,-0.020901,337,0,0,0,NaN,NaN,NaN
3,edp,edp_failure_file,T06,2017-12-29 21:20:00,3.4,27.2,0.013601,320.3,317.1,-3.2,...,1222.7,0.3,-0.005470,338,0,0,0,NaN,NaN,NaN
4,edp,edp_failure_file,T06,2017-12-29 21:50:00,3.3,27.6,0.013801,313.8,315.4,1.6,...,1200.5,0.3,-0.001990,347,0,0,0,NaN,NaN,NaN



merge_artifacts_summary.csv | colunas: 10


,artifact,rows,columns,failure_next_24h_positive,failure_next_72h_positive,failure_next_168h_positive,weak_failure_next_24h_positive,weak_failure_next_72h_positive,weak_failure_next_168h_positive,current_status_failure_like_positive
0,edp_gold_scada_failures_logs_metmast_empirical,209236,162,1295.0,3641.0,8248.0,NaN,NaN,NaN,NaN
1,penmanshiel_scada_status_weak_labels,737856,50,NaN,NaN,NaN,55156.0,135993.0,262622.0,5102.0
2,penmanshiel_status_failure_candidates,1267,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,edp_penmanshiel_common_schema,947092,23,56451.0,139634.0,270870.0,NaN,NaN,NaN,5102.0



penmanshiel_scada_status_weak_labels.csv | colunas: 50


,dataset,dataset_label,source_type,timestamp,entity,wind_speed_ms,power_kw,power_norm,wind_direction_deg,nac_direction_deg,...,weak_failure_next_Status,weak_failure_next_Code,weak_failure_next_Message,weak_failure_next_Service contract category,weak_failure_next_IEC category,weak_failure_next_weak_failure_type,weak_failure_next_24h,weak_failure_next_72h,weak_failure_next_168h,current_status_failure_like
0,penmanshiel,Penmanshiel SCADA 2020+,operational_scada_status,2020-01-01 00:00:00,Penmanshiel 01,2.627427,-4.735457,-0.002310,239.849839,231.364258,...,Stop,4510,Tower oscillation Y level 1,Safety stop of WEC (15),Forced outage,safety,0,0,1,0
1,penmanshiel,Penmanshiel SCADA 2020+,operational_scada_status,2020-01-01 00:10:00,Penmanshiel 01,2.513875,-2.004182,-0.000978,219.579114,231.364258,...,Stop,4510,Tower oscillation Y level 1,Safety stop of WEC (15),Forced outage,safety,0,0,1,0
2,penmanshiel,Penmanshiel SCADA 2020+,operational_scada_status,2020-01-01 00:20:00,Penmanshiel 01,1.909575,-2.267019,-0.001106,235.777395,231.364258,...,Stop,4510,Tower oscillation Y level 1,Safety stop of WEC (15),Forced outage,safety,0,0,1,0
3,penmanshiel,Penmanshiel SCADA 2020+,operational_scada_status,2020-01-01 00:30:00,Penmanshiel 01,1.269637,-1.756993,-0.000857,252.689586,231.364258,...,Stop,4510,Tower oscillation Y level 1,Safety stop of WEC (15),Forced outage,safety,0,0,1,0
4,penmanshiel,Penmanshiel SCADA 2020+,operational_scada_status,2020-01-01 00:40:00,Penmanshiel 01,0.273525,-1.430491,-0.000698,103.131663,231.364258,...,Stop,4510,Tower oscillation Y level 1,Safety stop of WEC (15),Forced outage,safety,0,0,1,0



penmanshiel_status_failure_candidates.csv | colunas: 13


,status_timestamp,status_end,Duration,Status,Code,Message,Comment,Service contract category,IEC category,entity,status_file,weak_failure_event,weak_failure_type
0,2020-01-07 14:03:28,2020-01-07 14:13:59,00:10:31,Stop,4510,Tower oscillation Y level 1,NaN,Safety stop of WEC (15),Forced outage,Penmanshiel 01,Status_Penmanshiel_01_2020-01-01_-_2021-01-01_...,1,safety
1,2020-01-07 15:28:48,2020-01-07 15:44:18,00:15:30,Stop,4530,Tower oscillation Y level 2,NaN,Safety stop of WEC (15),Forced outage,Penmanshiel 01,Status_Penmanshiel_01_2020-01-01_-_2021-01-01_...,1,safety
2,2020-01-07 15:47:19,2020-01-07 16:14:33,00:27:14,Stop,4510,Tower oscillation Y level 1,NaN,Safety stop of WEC (15),Forced outage,Penmanshiel 01,Status_Penmanshiel_01_2020-01-01_-_2021-01-01_...,1,safety
3,2020-01-11 00:21:02,2020-01-11 00:36:35,00:15:33,Stop,4540,Tower oscillation X level 2,NaN,Safety stop of WEC (15),Forced outage,Penmanshiel 01,Status_Penmanshiel_01_2020-01-01_-_2021-01-01_...,1,safety
4,2020-01-11 00:39:37,2020-01-11 00:55:08,00:15:31,Stop,4530,Tower oscillation Y level 2,NaN,Safety stop of WEC (15),Forced outage,Penmanshiel 01,Status_Penmanshiel_01_2020-01-01_-_2021-01-01_...,1,safety


## Observacoes para a proxima etapa

- O EDP deve ser tratado como ground truth principal.
- O Penmanshiel deve ser tratado como rotulo fraco, nao como verdade absoluta.
- As labels `failure_next_24h`, `failure_next_72h` e `failure_next_168h` devem ser avaliadas separadamente.
- Para modelagem, usar split temporal ou por turbina, evitando split aleatorio por linha.
